In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given a weighted directed graph of <code>N</code> vertices represented as an
  <code>N</code> &times; <code>N</code> distance matrix, compute the shortest path distance between
  every pair of vertices using the Floyd-Warshall algorithm. The matrix is stored as a flat array in
  row-major order: <code>dist[i * N + j]</code> is the weight of the directed edge from vertex
  <code>i</code> to vertex <code>j</code>. A value of <code>+infinity</code> means no direct edge
  exists. The diagonal is always zero. For each intermediate vertex <code>k</code> from <code>0</code> to <code>N - 1</code>
  (in order), update all pairs:
</p>
<p>
  $$
    \text{output}[i][j] = \min\!\bigl(\text{output}[i][j],\;
      \text{output}[i][k] + \text{output}[k][j]\bigr)
    \quad \forall\, i, j
  $$
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>output</code></li>
</ul>

<h2>Example:</h2>
<pre>
Input: N = 4
dist = [
  0,   5, inf,  10,   // row 0: edges from vertex 0
  inf, 0,   3, inf,   // row 1: edges from vertex 1
  inf, inf, 0,   1,   // row 2: edges from vertex 2
  inf, inf, inf, 0    // row 3: edges from vertex 3
]

Output:
output = [
  0,   5,   8,   9,   // shortest paths from vertex 0
  inf, 0,   3,   4,   // shortest paths from vertex 1
  inf, inf, 0,   1,   // shortest paths from vertex 2
  inf, inf, inf, 0    // shortest paths from vertex 3
]

Explanation:
- output[0][2] = 8   (path 0 -&gt; 1 -&gt; 2, cost 5 + 3 = 8)
- output[0][3] = 9   (path 0 -&gt; 1 -&gt; 2 -&gt; 3, cost 5 + 3 + 1 = 9, beats direct 0 -&gt; 3 = 10)
- output[1][3] = 4   (path 1 -&gt; 2 -&gt; 3, cost 3 + 1 = 4)
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 4,096</li>
  <li>Edge weights are finite <code>float32</code> values or <code>+infinity</code> (no edge)</li>
  <li>The input contains no negative cycles</li>
  <li>The diagonal satisfies <code>dist[i * N + i] = 0</code> for all <code>i</code></li>
  <li><code>dist</code> and <code>output</code> are flat arrays of <code>N &times; N</code> floats in row-major order</li>
  <li>Performance is measured with <code>N</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// dist, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* dist, float* output, int N) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# dist, output are tensors on the GPU
@cute.jit
def solve(dist: cute.Tensor, output: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# dist is a tensor on the GPU
@jax.jit
def solve(dist: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# dist, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    dist: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# dist, output are tensors on the GPU
def solve(dist: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# dist, output are tensors on the GPU
def solve(dist: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/hard/73_all_pairs_shortest_paths/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
